In [ ]:
import os
import pandas as pd
import sys
from tqdm import tqdm
os.environ['PATH_TO_REPO'] = '/Users/stevie/repos/lingo_kit_super/lingo_kit_data'

In [ ]:
sys.path.append('/Users/stevie/repos/lingo_kit_super/lingo_kit_data/utils/audio')
from text_to_speech import get_audio_hash, VOICES, get_duration_ms
sys.path.append('/Users/stevie/repos/lingo_kit_super/lingo_kit_data/utils/s3')
from upload_to_s3 import already_uploaded

In [ ]:
en_speaking_rate = 0.92
it_speaking_rate = 0.7

In [ ]:
# path = '/Users/stevie/repos/lingo_kit_super/lingo_kit_data/vocabulary/dataframes/VERB/essere.tsv'
path = '/Users/stevie/repos/lingo_kit_super/lingo_kit_data/vocabulary/dataframes/DET/nue.tsv'
df = pd.read_csv(path, sep='\t')
row = df.iloc[0]

In [ ]:
# get_audio_hash(text, voice_name, speaking_rate, pitch)
en_audio_hash = get_audio_hash(
    text=row['translation_en'],
    # voice_name=VOICES['english']['male'],
    voice_name=VOICES['english']['female'],
    speaking_rate=en_speaking_rate,
    pitch=0,
)
print(en_audio_hash)
print(f"{en_audio_hash}.mp3" in already_uploaded)

In [ ]:
# get_audio_hash(text, voice_name, speaking_rate, pitch)
it_audio_hash = get_audio_hash(
    text=row['text'],
    # voice_name=VOICES['english']['male'],
    voice_name=VOICES['italian']['male'],
    speaking_rate=it_speaking_rate,
    pitch=0,
)
print(it_audio_hash)
print(f"{it_audio_hash}.mp3" in already_uploaded)

In [ ]:
df_dir = '/Users/stevie/repos/lingo_kit_super/lingo_kit_data/vocabulary/dataframes'
data = {
    'it_found_count': [],
    'en_found_count': [],
    'total_count': [],
}
pos_folders = os.listdir(df_dir)
for pos_folder in pos_folders:
    print(pos_folder)
    pos_dir = os.path.join(df_dir, pos_folder)
    lemma_files = os.listdir(pos_dir)

    for lemma_file in tqdm(lemma_files):
        lemma_path = os.path.join(pos_dir, lemma_file)
        df = pd.read_csv(lemma_path, sep='\t')
        found_it = 0
        found_en = 0

        for i, row in df.iterrows():

            it_audio_hash = get_audio_hash(
                text=row['text'],
                # voice_name=VOICES['english']['male'],
                voice_name=VOICES['italian']['male'],
                speaking_rate=it_speaking_rate,
                pitch=0,
            )
            if f"{it_audio_hash}.mp3" in already_uploaded:
                found_it += 1

            else:
                it_audio_hash = get_audio_hash(
                    text=row['text'],
                    # voice_name=VOICES['italian']['male'],
                    voice_name=VOICES['italian']['female'],
                    speaking_rate=it_speaking_rate,
                    pitch=0,
                )
                if f"{it_audio_hash}.mp3" in already_uploaded:
                    found_it += 1
                

            en_audio_hash = get_audio_hash(
                text=row['translation_en'],
                # voice_name=VOICES['english']['male'],
                voice_name=VOICES['english']['female'],
                speaking_rate=en_speaking_rate,
                pitch=0,
            )
            if f"{en_audio_hash}.mp3" in already_uploaded:
                found_en += 1
            else:
                en_audio_hash = get_audio_hash(
                    text=row['translation_en'],
                    voice_name=VOICES['english']['male'],
                    # voice_name=VOICES['english']['female'],
                    speaking_rate=en_speaking_rate,
                    pitch=0,
                )
                if f"{en_audio_hash}.mp3" in already_uploaded:
                    found_en += 1
        
        data['it_found_count'].append(found_it)
        data['en_found_count'].append(found_en)
        data['total_count'].append(len(df))
result_df = pd.DataFrame(data)

In [ ]:
result_df['equal_en_it'] = result_df['it_found_count'] == result_df['en_found_count']
result_df['equal_en_total'] = result_df['en_found_count'] == result_df['total_count']
result_df['equal_it_total'] = result_df['it_found_count'] == result_df['total_count']

In [ ]:
# equal_en_it
# True     12488
# False     1119
# Name: count, dtype: int64
result_df['equal_en_it'].value_counts()

In [ ]:
# equal_en_total
# False    12910
# True       697
# Name: count, dtype: int64
result_df['equal_en_total'].value_counts()

In [ ]:
# equal_it_total
# False    12425
# True      1182
# Name: count, dtype: int64
result_df['equal_it_total'].value_counts()

In [ ]:
df_dir = '/Users/stevie/repos/lingo_kit_super/lingo_kit_data/vocabulary/dataframes'
pos_folders = os.listdir(df_dir)
for pos_folder in pos_folders:
    print(pos_folder)
    pos_dir = os.path.join(df_dir, pos_folder)
    lemma_files = os.listdir(pos_dir)

    for lemma_file in tqdm(lemma_files):
        lemma_path = os.path.join(pos_dir, lemma_file)
        df = pd.read_csv(lemma_path, sep='\t')

        for i, row in df.iterrows():

            it_audio_hash = get_audio_hash(
                text=row['text'],
                voice_name=VOICES['italian']['male'],
                speaking_rate=it_speaking_rate,
                pitch=0,
            )

            en_audio_hash = get_audio_hash(
                text=row['translation_en'],
                voice_name=VOICES['english']['female'],
                speaking_rate=en_speaking_rate,
                pitch=0,
            )

            df.loc[i, 'italian_audio_hash'] = it_audio_hash
            df.loc[i, 'english_audio_hash'] = en_audio_hash

            # if the audio files exist, then calculate their duration ms
            it_file = f"/Users/stevie/repos/lingo_kit_super/lingo_kit_data/data/audio/{it_audio_hash}.mp3"
            en_file = f"/Users/stevie/repos/lingo_kit_super/lingo_kit_data/data/audio/{en_audio_hash}.mp3"

            it_dur = None
            en_dur = None
            if os.path.exists(it_file):
                it_dur = get_duration_ms(it_file)
            if os.path.exists(en_file):
                en_dur = get_duration_ms(en_file)

            df.loc[i, 'italian_audio_duration_ms'] = it_dur
            df.loc[i, 'english_audio_duration_ms'] = en_dur

        df.to_csv(lemma_path, sep='\t', index=False)